# Phase 1 – Data Acquisition

This notebook downloads, validates, and stores all datasets used in the project.

No statistical analysis, rolling diagnostics, or collapse identification is performed here.

Outputs:

- Raw factor data
- Raw VIX data
- Cleaned daily factor returns
- Metadata and checksums

In [16]:
import hashlib
import io
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd
import requests

In [17]:
from google.colab import drive
drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [18]:
from pathlib import Path

ROOT = Path(
    "/content/drive/MyDrive/When_Do_Factor_Premia_Break"
)

RAW = ROOT / "data" / "raw"
PROCESSED = ROOT / "data" / "processed"

RAW.mkdir(parents=True, exist_ok=True)
PROCESSED.mkdir(parents=True, exist_ok=True)

print("Project root:", ROOT)
print("Raw data:", RAW)
print("Processed data:", PROCESSED)

Project root: /content/drive/MyDrive/When_Do_Factor_Premia_Break
Raw data: /content/drive/MyDrive/When_Do_Factor_Premia_Break/data/raw
Processed data: /content/drive/MyDrive/When_Do_Factor_Premia_Break/data/processed


In [19]:
import os

os.chdir(ROOT)
print("Current working directory:", Path.cwd())

Current working directory: /content/drive/MyDrive/When_Do_Factor_Premia_Break


In [20]:
FF5_URL = (
    "https://mba.tuck.dartmouth.edu/pages/faculty/"
    "ken.french/ftp/F-F_Research_Data_5_Factors_2x3_daily_CSV.zip"
)

MOM_URL = (
    "https://mba.tuck.dartmouth.edu/pages/faculty/"
    "ken.french/ftp/F-F_Momentum_Factor_daily_CSV.zip"
)

VIX_URL = (
    "https://cdn.cboe.com/api/global/us_indices/daily_prices/"
    "VIX_History.csv"
)

In [21]:
def download_file(url, destination):
    r = requests.get(url)

    r.raise_for_status()

    with open(destination, "wb") as f:
        f.write(r.content)

    print(f"Downloaded {destination.name}")

In [22]:
download_file(
    FF5_URL,
    RAW / "ff5_daily.zip"
)

download_file(
    MOM_URL,
    RAW / "momentum.zip"
)

download_file(
    VIX_URL,
    RAW / "vix.csv"
)

Downloaded ff5_daily.zip
Downloaded momentum.zip
Downloaded vix.csv


In [23]:
def sha256(path):

    h = hashlib.sha256()

    with open(path, "rb") as f:

        while chunk := f.read(8192):
            h.update(chunk)

    return h.hexdigest()

checksums = {
    path.name: sha256(path)
    for path in RAW.iterdir()
    if path.is_file() and path.name != "checksums.csv"
}

checksum_df = (
    pd.Series(checksums, name="sha256")
    .rename_axis("filename")
    .reset_index()
)

checksum_df.to_csv(
    RAW / "checksums.csv",
    index=False
)

checksum_df

,filename,sha256
0,ff5_daily.zip,bcf32ecc9e2bb20383784ac98891e42146a0091eec6ec7...
1,momentum.zip,f4237e2e36dffa13fd7823f55376316a94b5ac663af951...
2,vix.csv,f79850788eeb4187019667b854187d386ffdfab177ad94...


## Download the Source Data

This section downloads the datasets used throughout the project.

Sources:

- Fama-French 5 Factors (Daily)
- Fama-French Momentum Factor (Daily)
- CBOE VIX Daily History

These files are saved in the `data/raw` directory without modification.

In [24]:
# Ken French Daily 5-Factor Dataset
FF5_URL = (
    "https://mba.tuck.dartmouth.edu/pages/faculty/"
    "ken.french/ftp/F-F_Research_Data_5_Factors_2x3_daily_CSV.zip"
)

# Ken French Daily Momentum Factor
MOM_URL = (
    "https://mba.tuck.dartmouth.edu/pages/faculty/"
    "ken.french/ftp/F-F_Momentum_Factor_daily_CSV.zip"
)

# CBOE VIX Daily History
VIX_URL = (
    "https://cdn.cboe.com/api/global/us_indices/"
    "daily_prices/VIX_History.csv"
)

In [25]:
import requests

def download_file(url: str, destination):
    """
    Download a file if it doesn't already exist.
    """

    if destination.exists():
        print(f"✓ {destination.name} already exists")
        return

    response = requests.get(url, timeout=60)
    response.raise_for_status()

    with open(destination, "wb") as f:
        f.write(response.content)

    print(f"✓ Downloaded {destination.name}")

In [26]:
download_file(
    FF5_URL,
    RAW / "ff5_daily.zip"
)

download_file(
    MOM_URL,
    RAW / "momentum_daily.zip"
)

download_file(
    VIX_URL,
    RAW / "vix_history.csv"
)

✓ ff5_daily.zip already exists
✓ Downloaded momentum_daily.zip
✓ Downloaded vix_history.csv


In [27]:
from pathlib import Path

for file in RAW.iterdir():
    if file.is_file():
        size_mb = file.stat().st_size / (1024 * 1024)
        print(f"{file.name:<25} {size_mb:.2f} MB")

ff5_daily.zip             0.14 MB
momentum.zip              0.09 MB
vix.csv                   0.45 MB
checksums.csv             0.00 MB
momentum_daily.zip        0.09 MB
vix_history.csv           0.45 MB


##Parse and Clean Ken French Factor Data

The Ken French source files contain descriptive text above and below the actual
data table. This section identifies valid daily observations by locating rows
whose first field is an eight-digit date in `YYYYMMDD` format.

Returns are published in percentage units and are converted to decimal units.
No rolling statistics, event labels, or diagnostic signals are calculated here.

In [34]:
import io
import re
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd

In [35]:
def inspect_zip(zip_path: Path) -> list[str]:
    """Return the filenames contained in a ZIP archive."""
    with zipfile.ZipFile(zip_path, "r") as archive:
        return archive.namelist()


print("Five-factor archive:")
print(inspect_zip(RAW / "ff5_daily.zip"))

print("\nMomentum archive:")
print(inspect_zip(RAW / "momentum_daily.zip"))

Five-factor archive:
['F-F_Research_Data_5_Factors_2x3_daily.csv']

Momentum archive:
['F-F_Momentum_Factor_daily.csv']


In [36]:
def read_text_from_zip(zip_path: Path) -> str:
    """
    Read the main CSV or TXT file contained in a ZIP archive.

    The function ignores directories and selects the first file ending in
    .csv, .txt, or .dat. If none match, it selects the first regular file.
    """
    with zipfile.ZipFile(zip_path, "r") as archive:
        members = [
            name
            for name in archive.namelist()
            if not name.endswith("/")
        ]

        if not members:
            raise ValueError(f"No files found inside {zip_path.name}")

        preferred = [
            name
            for name in members
            if name.lower().endswith((".csv", ".txt", ".dat"))
        ]

        selected = preferred[0] if preferred else members[0]

        raw_bytes = archive.read(selected)

    # Ken French files are generally compatible with this encoding.
    return raw_bytes.decode("utf-8-sig", errors="replace")

ff5_text = read_text_from_zip(RAW / "ff5_daily.zip")
mom_text = read_text_from_zip(RAW / "momentum_daily.zip")

print("Five-factor text length:", len(ff5_text))
print("Momentum text length:", len(mom_text))

Five-factor text length: 1013735
Momentum text length: 427515


In [37]:
DAILY_DATE_PATTERN = re.compile(r"^\s*(\d{8})\s*,")


def extract_daily_table(text: str) -> str:
    """
    Extract consecutive daily data rows from a Ken French text file.

    A valid row must begin with an eight-digit YYYYMMDD date followed by
    a comma. Descriptive headers, annual tables, footnotes, and blank lines
    are excluded.
    """
    daily_rows = []

    for line in text.splitlines():
        if DAILY_DATE_PATTERN.match(line):
            daily_rows.append(line.strip())

    if not daily_rows:
        raise ValueError("No daily rows were detected in the source file.")

    return "\n".join(daily_rows)

ff5_table_text = extract_daily_table(ff5_text)
mom_table_text = extract_daily_table(mom_text)

print("Five-factor rows:", len(ff5_table_text.splitlines()))
print("Momentum rows:", len(mom_table_text.splitlines()))

Five-factor rows: 15833
Momentum rows: 26152


In [38]:
FF5_COLUMNS = [
    "date",
    "MKT_RF",
    "SMB",
    "HML",
    "RMW",
    "CMA",
    "RF",
]


def parse_ff5_daily(zip_path: Path) -> pd.DataFrame:
    """Parse the daily Fama-French five-factor ZIP file."""
    text = read_text_from_zip(zip_path)
    table_text = extract_daily_table(text)

    dataframe = pd.read_csv(
        io.StringIO(table_text),
        header=None,
        names=FF5_COLUMNS,
        skipinitialspace=True,
    )

    dataframe["date"] = pd.to_datetime(
        dataframe["date"].astype(str).str.strip(),
        format="%Y%m%d",
        errors="raise",
    )

    factor_columns = [column for column in FF5_COLUMNS if column != "date"]

    for column in factor_columns:
        dataframe[column] = pd.to_numeric(
            dataframe[column],
            errors="coerce",
        )

    # Ken French returns are supplied as percentages.
    dataframe[factor_columns] = dataframe[factor_columns] / 100.0

    return dataframe.set_index("date").sort_index()

ff5 = parse_ff5_daily(RAW / "ff5_daily.zip")

print("Shape:", ff5.shape)
print("Columns:", ff5.columns.tolist())
print("Date range:", ff5.index.min(), "to", ff5.index.max())

ff5.head()

Shape: (15833, 6)
Columns: ['MKT_RF', 'SMB', 'HML', 'RMW', 'CMA', 'RF']
Date range: 1963-07-01 00:00:00 to 2026-05-29 00:00:00


,MKT_RF,SMB,HML,RMW,CMA,RF
date,,,,,,
1963-07-01,-0.0067,0.0000,-0.0034,-0.0001,0.0016,0.0001
1963-07-02,0.0079,-0.0026,0.0026,-0.0007,-0.0020,0.0001
1963-07-03,0.0063,-0.0017,-0.0009,0.0018,-0.0034,0.0001
1963-07-05,0.0040,0.0008,-0.0027,0.0009,-0.0034,0.0001
1963-07-08,-0.0063,0.0004,-0.0018,-0.0029,0.0014,0.0001


In [39]:
def parse_momentum_daily(zip_path: Path) -> pd.DataFrame:
    """
    Parse the daily Ken French momentum-factor ZIP file.

    The source may contain a trailing empty column, so only the first
    two fields—date and momentum return—are imported.
    """
    text = read_text_from_zip(zip_path)
    table_text = extract_daily_table(text)

    dataframe = pd.read_csv(
        io.StringIO(table_text),
        header=None,
        usecols=[0, 1],
        names=["date", "MOM"],
        skipinitialspace=True,
    )

    dataframe["date"] = pd.to_datetime(
        dataframe["date"].astype(str).str.strip(),
        format="%Y%m%d",
        errors="raise",
    )

    dataframe["MOM"] = pd.to_numeric(
        dataframe["MOM"],
        errors="raise",
    )

    # Ken French publishes returns in percentage units.
    dataframe["MOM"] = dataframe["MOM"] / 100.0

    return (
        dataframe
        .set_index("date")
        .sort_index()
    )


momentum = parse_momentum_daily(
    RAW / "momentum_daily.zip"
)

print("Shape:", momentum.shape)
print("Columns:", momentum.columns.tolist())
print(
    "Date range:",
    momentum.index.min(),
    "to",
    momentum.index.max(),
)

momentum.head()


Shape: (26152, 1)
Columns: ['MOM']
Date range: 1926-11-03 00:00:00 to 2026-05-29 00:00:00


,MOM
date,
1926-11-03,0.0035
1926-11-04,-0.0061
1926-11-05,0.0115
1926-11-06,-0.0018
1926-11-08,-0.0005


In [40]:
def basic_validation(
    dataframe: pd.DataFrame,
    expected_columns: list[str],
    name: str,
) -> None:
    """Run basic structural checks on a parsed dataset."""
    print(f"Validating {name}...")

    assert isinstance(dataframe.index, pd.DatetimeIndex), (
        f"{name}: index is not a DatetimeIndex"
    )

    assert dataframe.index.is_monotonic_increasing, (
        f"{name}: dates are not sorted"
    )

    assert dataframe.index.is_unique, (
        f"{name}: duplicate dates detected"
    )

    assert dataframe.columns.tolist() == expected_columns, (
        f"{name}: unexpected columns: {dataframe.columns.tolist()}"
    )

    assert len(dataframe) > 0, (
        f"{name}: dataset is empty"
    )

    assert not dataframe.isna().all(axis=1).any(), (
        f"{name}: fully missing rows detected"
    )

    print(f"✓ {name} passed basic validation")

basic_validation(
    ff5,
    expected_columns=[
        "MKT_RF",
        "SMB",
        "HML",
        "RMW",
        "CMA",
        "RF",
    ],
    name="Fama-French five-factor data",
)

basic_validation(
    momentum,
    expected_columns=["MOM"],
    name="Momentum data",
)

Validating Fama-French five-factor data...
✓ Fama-French five-factor data passed basic validation
Validating Momentum data...
✓ Momentum data passed basic validation


In [41]:
print("Five-factor missing values:")
display(ff5.isna().sum().to_frame("missing_count"))

print("\nMomentum missing values:")
display(momentum.isna().sum().to_frame("missing_count"))

Five-factor missing values:


,missing_count
MKT_RF,0
SMB,0
HML,0
RMW,0
CMA,0
RF,0



Momentum missing values:


,missing_count
MOM,0


In [42]:
def return_range_check(dataframe: pd.DataFrame) -> pd.DataFrame:
    """Summarize value ranges for unit validation."""
    return pd.DataFrame(
        {
            "minimum": dataframe.min(),
            "maximum": dataframe.max(),
            "median_abs": dataframe.abs().median(),
        }
    )


print("Five-factor range check:")
display(return_range_check(ff5))

print("\nMomentum range check:")
display(return_range_check(momentum))

Five-factor range check:


,minimum,maximum,median_abs
MKT_RF,-0.1744,0.1136,0.0047
SMB,-0.1115,0.0608,0.0029
HML,-0.0503,0.0673,0.0025
RMW,-0.0297,0.0457,0.0019
CMA,-0.0531,0.0248,0.0019
RF,0.0000,0.0006,0.0002



Momentum range check:


,minimum,maximum,median_abs
MOM,-0.1823,0.0712,0.0031


In [43]:
factors = ff5.join(
    momentum,
    how="inner",
)

FACTOR_COLUMNS = [
    "MKT_RF",
    "SMB",
    "HML",
    "RMW",
    "CMA",
    "MOM",
]

factors = factors[
    FACTOR_COLUMNS + ["RF"]
].copy()

print("Merged shape:", factors.shape)
print(
    "Merged date range:",
    factors.index.min(),
    "to",
    factors.index.max(),
)

factors.head()

Merged shape: (15833, 7)
Merged date range: 1963-07-01 00:00:00 to 2026-05-29 00:00:00


,MKT_RF,SMB,HML,RMW,CMA,MOM,RF
date,,,,,,,
1963-07-01,-0.0067,0.0000,-0.0034,-0.0001,0.0016,-0.0024,0.0001
1963-07-02,0.0079,-0.0026,0.0026,-0.0007,-0.0020,0.0044,0.0001
1963-07-03,0.0063,-0.0017,-0.0009,0.0018,-0.0034,0.0038,0.0001
1963-07-05,0.0040,0.0008,-0.0027,0.0009,-0.0034,0.0006,0.0001
1963-07-08,-0.0063,0.0004,-0.0018,-0.0029,0.0014,-0.0045,0.0001


In [44]:
assert isinstance(factors.index, pd.DatetimeIndex)
assert factors.index.is_monotonic_increasing
assert factors.index.is_unique

assert factors.columns.tolist() == [
    "MKT_RF",
    "SMB",
    "HML",
    "RMW",
    "CMA",
    "MOM",
    "RF",
]

assert not factors.empty
assert not factors.isna().any().any()

print("✓ Merged factor dataset passed validation")

✓ Merged factor dataset passed validation


In [45]:
validation_report = pd.DataFrame(
    {
        "rows": [len(factors)],
        "start_date": [factors.index.min()],
        "end_date": [factors.index.max()],
        "duplicate_dates": [
            int(factors.index.duplicated().sum())
        ],
        "missing_values": [
            int(factors.isna().sum().sum())
        ],
        "number_of_columns": [
            len(factors.columns)
        ],
    }
)

validation_report

,rows,start_date,end_date,duplicate_dates,missing_values,number_of_columns
0,15833,1963-07-01,2026-05-29,0,0,7


In [46]:
FACTOR_OUTPUT = (
    PROCESSED / "factor_returns_daily.parquet"
)

factors.to_parquet(
    FACTOR_OUTPUT,
    index=True,
)

print(f"Saved: {FACTOR_OUTPUT}")

VALIDATION_OUTPUT = (
    PROCESSED / "factor_data_validation.csv"
)

validation_report.to_csv(
    VALIDATION_OUTPUT,
    index=False,
)

print(f"Saved: {VALIDATION_OUTPUT}")

Saved: /content/drive/MyDrive/When_Do_Factor_Premia_Break/data/processed/factor_returns_daily.parquet
Saved: /content/drive/MyDrive/When_Do_Factor_Premia_Break/data/processed/factor_data_validation.csv


In [47]:
factors_reloaded = pd.read_parquet(
    FACTOR_OUTPUT
)

pd.testing.assert_frame_equal(
    factors,
    factors_reloaded,
)

print("✓ Saved and reloaded dataset matches exactly")

✓ Saved and reloaded dataset matches exactly


In [48]:
print("Processed files:")

for path in sorted(PROCESSED.iterdir()):
    if path.is_file():
        size_kb = path.stat().st_size / 1024
        print(f"{path.name:<35} {size_kb:,.1f} KB")

Processed files:
factor_data_validation.csv          0.1 KB
factor_returns_daily.parquet        272.1 KB


## Parse and Validate VIX Data

This section loads the CBOE VIX daily history, validates its structure, and
stores a cleaned version for later comparison.

No analyses involving VIX are performed in this notebook.

In [49]:
vix = pd.read_csv(
    RAW / "vix_history.csv"
)

print(vix.shape)

vix.head()

(9240, 5)


,DATE,OPEN,HIGH,LOW,CLOSE
0,01/02/1990,17.24,17.24,17.24,17.24
1,01/03/1990,18.19,18.19,18.19,18.19
2,01/04/1990,19.22,19.22,19.22,19.22
3,01/05/1990,20.11,20.11,20.11,20.11
4,01/08/1990,20.26,20.26,20.26,20.26


In [50]:
vix.columns = (
    vix.columns
       .str.strip()
       .str.upper()
)

vix.head()

,DATE,OPEN,HIGH,LOW,CLOSE
0,01/02/1990,17.24,17.24,17.24,17.24
1,01/03/1990,18.19,18.19,18.19,18.19
2,01/04/1990,19.22,19.22,19.22,19.22
3,01/05/1990,20.11,20.11,20.11,20.11
4,01/08/1990,20.26,20.26,20.26,20.26


In [51]:
vix["DATE"] = pd.to_datetime(
    vix["DATE"],
    errors="raise"
)

vix = (
    vix
    .sort_values("DATE")
    .set_index("DATE")
)

In [52]:
vix = vix[["CLOSE"]]

vix = vix.rename(
    columns={
        "CLOSE": "VIX"
    }
)

vix.head()

,VIX
DATE,
1990-01-02,17.24
1990-01-03,18.19
1990-01-04,19.22
1990-01-05,20.11
1990-01-08,20.26


In [53]:
assert isinstance(
    vix.index,
    pd.DatetimeIndex
)

assert vix.index.is_unique
assert vix.index.is_monotonic_increasing

assert not vix.isna().any().any()

assert (vix["VIX"] > 0).all()

print("✓ VIX passed validation")
vix.describe()

✓ VIX passed validation


,VIX
count,9240.000000
mean,19.441970
std,7.729515
min,9.140000
25%,13.980000
50%,17.610000
75%,22.700000
max,82.690000


In [54]:
VIX_OUTPUT = (
    PROCESSED / "vix_daily.parquet"
)

vix.to_parquet(
    VIX_OUTPUT
)

print(f"Saved: {VIX_OUTPUT}")

Saved: /content/drive/MyDrive/When_Do_Factor_Premia_Break/data/processed/vix_daily.parquet


In [55]:
vix_reload = pd.read_parquet(
    VIX_OUTPUT
)

pd.testing.assert_frame_equal(
    vix,
    vix_reload
)

print("✓ Reload successful")

✓ Reload successful


In [56]:
print("=" * 60)
print("DATA ACQUISITION COMPLETE")
print("=" * 60)

print(f"Factor observations : {len(factors):,}")
print(f"Factor columns      : {list(factors.columns)}")
print()

print(f"VIX observations    : {len(vix):,}")
print()

print("Processed files:")
for file in PROCESSED.iterdir():
    print(" -", file.name)

DATA ACQUISITION COMPLETE
Factor observations : 15,833
Factor columns      : ['MKT_RF', 'SMB', 'HML', 'RMW', 'CMA', 'MOM', 'RF']

VIX observations    : 9,240

Processed files:
 - factor_returns_daily.parquet
 - factor_data_validation.csv
 - vix_daily.parquet
